# MCP

In [7]:
from setup import bedrock_tool
import os
import chromadb
from agents import Agent, Runner, function_tool, trace
from agents.mcp import MCPServerStreamableHttp

Let's set up our RAG database connection:

In [8]:
chroma_client = chromadb.PersistentClient(path="../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [9]:
# This is the same code as in the RAG notebook


@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Integrate EXA Search as an MCP:

In [10]:
# Increased EXA timeout from 30 to 90 seconds since EXA can take longer than 30 seconds to respond when under heavy load
exa_search_mcp = MCPServerStreamableHttp(
    name="Exa Search MCP",
    params={
        "url": f"https://mcp.exa.ai/mcp?exaApiKey={os.environ.get('EXA_API_KEY')}",
        "timeout": 90,
    },
    client_session_timeout_seconds=90,
    cache_tools_list=True,
    max_retry_attempts=1,
)

await exa_search_mcp.connect()

calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the EXA web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if necessary, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use Exa Search to find the exact recipe and ingredients.
    * Once you know the ingredients, use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 10 times.
    * Respond with Markdown formatting, and use bullet points and where appropriate. Don't use tables
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(calorie_lookup_tool.__dict__)],
    mcp_servers=[exa_search_mcp],
)

Reference query - shouldn't use ExaSearch:

In [11]:
with trace("Nutrition Assistant with MCP - Only uses calorie_lookup_tool") as t:
    result = await Runner.run(
        calorie_agent_with_search,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )

print(f"# Output:\n\n{result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

# Output:

- **Calories in a banana and an apple:**

  - **Banana:**
    - Calories per 100g: **89.0 calories**
    - For a single serving (118 grams): **105 calories**
    
  - **Apple:**
    - Calories per 100g: **52.0 calories**
    - For a single serving (185 grams): **95 calories**

- **Total calories in a banana and an apple:**
  - **Banana:** 105 calories
  - **Apple:** 95 calories
  - **Total:** 200 calories

- **Calories per 100g for each fruit:**
  - **Banana:** 89.0 calories per 100g
  - **Apple:** 52.0 calories per 100g

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_341d928601a74e8fa3d6f0258f95fcfb


In [12]:
with trace("Nutrition Assistant with MCP ") as t:
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast?"
    )

print(f"# Output:\n\n{result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

# Output:

The Full English Breakfast typically includes the following ingredients:

- **2 eggs:** 185 calories per 100g (assuming a duck egg for variety)
- **2-3 slices of bacon:** 107 calories per 100g
- **2 sausages:** 315 calories per 100g (assuming spam for variety)
- **1 tomato:** 18 calories per 100g
- **1/2 cup baked beans:** 94 calories per 100g
- **4-5 mushrooms:** 22 calories per 100g
- **2 slices of toast:** 361 calories per 100g

### Total Calories for a Single Serving:

- **Eggs:** 370 calories (for 2 eggs)
- **Bacon:** 321 calories (for 3 slices)
- **Sausages:** 630 calories (for 2 sausages)
- **Tomato:** 9 calories (for 1 tomato)
- **Baked Beans:** 47 calories (for 1/2 cup)
- **Mushrooms:** 11 calories (for 4-5 mushrooms)
- **Toast:** 722 calories (for 2 slices)

**Total Calories:** 2260 calories

Please note that these are estimated values based on the provided sources and the calorie information for the ingredients. The actual calorie count may vary depending on the s